### COT Report data scraper

https://github.com/NDelventhal/cot_reports

In [52]:
import pandas as pd
import cot_reports as cot
from matplotlib import pyplot as plt
import numpy as np
import re
#%matplotlib widget
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML


### <u> 1. Instructions for using cot_reports </u>


From https://github.com/NDelventhal/cot_reports

#### Example: cot_hist()
df = cot.cot_hist(cot_report_type= 'traders_in_financial_futures_futopt')
#### cot_hist() downloads the historical bulk file for the specified report type, in this example the Traders in Financial Futures Futures-and-Options Combined report. Returns the data as dataframe.

#### Example: cot_year()
df = cot.cot_year(year = 2020, cot_report_type = 'traders_in_financial_futures_fut')
#### cot_year() downloads the single year file of the specified report type and year. Returns the data as dataframe.

#### Example for collecting data of a few years, here from 2017 to 2020, of a specified report:
df = pd.DataFrame()
begin_year = 2017
end_year = 2020
for i in range(begin_year, end_year + 1):
    single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_futopt')) 
    df = df.append(single_year, ignore_index=True)

#### Example: cot_all()
df = cot.cot_all(cot_report_type='legacy_fut')
#### cot_all() downloads the historical bulk file and all remaining single year files of the specified report type.  Returns the data as dataframe.

### 2. Download and Compile COT Data into a pandas dataframe

In [53]:
def cot_reader (start, end):
    df_list = []
    begin_year = start
    end_year = end
    for i in range(begin_year, end_year + 1):
        single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_fut')) 
        df_list.append(single_year)
    
    df = pd.concat(df_list, ignore_index=True)
    
    df.rename(columns = {"Market and Exchange Names" : "Market", 
                     "As of Date in Form YYMMDD" : "Datetime", 
                     "Open Interest (All)" : "OI", 
                     "As of Date in Form YYYY-MM-DD" : "Date"}, inplace=True )
    
    df["Datetime"] = pd.to_datetime(df["Datetime"], format = '%y%m%d')
    
    df.sort_values("Datetime", ascending = False, inplace = True)
    
    return df

In [54]:
df = cot_reader(2022, 2026)

Selected: legacy_fut
Downloaded single year data from: 2022
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2023
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2024
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2025
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2026
Stored the file annual.txt in the working directory.


### 3. Create a list of markets to trade 

##### This is based on personal preference, please ignore if you're looking at all markets or edit as you like

In [55]:
keywords = ["gold", "silver", "platinum", "palladium", "copper", "Lithium", "crude", "heating", "oil", "Nat Gas",
"rbob", "brent", "cocoa", "corn", "oat", "wheat", "soybean", "soy bean", "feed" , "Hogs", "live", "OJ", "coffee", "cotton", "sugar","E-Mini", "Micro","WTI-PHYSICAL",
    "Russell",
    "S&P 500",
    "NASDAQ",
    "Dow Jones",
    "Nikkei",
    "FTSE",
    "DAX",
    "CAC",
    "SMI",
    "Hang Seng",
    "Shanghai",
    "Treasury", "UST", "Bond", "EURO" , "Peso", "Brazilian", "Swiss", "Canadian", "British", "Japanese", "New Zealand", "Rand", 
    "bitcoin" , "ether","SOFR", "vix"
           ]

In [56]:
unique_markets = []

for keyword in keywords:
    filtered_df = df[df["Market"].str.contains(keyword, case=False)]
    
    unique_values = filtered_df["Market"].unique()
    
    unique_markets.extend(unique_values)
    

In [57]:
remove_items = [
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    'EURO SHORT TERM RATE - CHICAGO MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTY-PAID - COMMODITY EXCHANGE INC.',
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH EURODOLLARS - CHICAGO MERCANTILE EXCHANGE',
    'COFFEE CALENDAR SPREAD OPTIONS - ICE FUTURES U.S.',
    'WHEAT-HRW - CHICAGO BOARD OF TRADE',
    'WHEAT-HRSpring - MINNEAPOLIS GRAIN EXCHANGE',
    'BLACK SEA WHEAT FINANCIAL - CHICAGO BOARD OF TRADE',
    'CORN CONSECUTIVE CSO - CHICAGO BOARD OF TRADE',
    'CORN CSO - CHICAGO BOARD OF TRADE',
    'MARINE .5% FOB USGC/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'WTI-BRENT SPREAD OPTION - NEW YORK MERCANTILE EXCHANGE',
    'WTI-BRENT CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'USGC HSFO-PLATTS/BRENT 1ST LN - ICE FUTURES ENERGY DIV',
    'TRANSCONTINENTAL GAS- STATION 85 (ZONE 4) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - VENTURA (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-TEXOK (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (BASIS) - ICE FUTURES ENERGY DIV',
    'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COLUMBIA GAS CO. - TCO POOL (APPALACHIA) (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-MID-CONTINENT POOL PIN (BASIS) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - DEMARCATION POOL (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (INDEX) - ICE FUTURES ENERGY DIV',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'ONEOK GAS TRANSPORTATION BASIS - ICE FUTURES ENERGY DIV',
    'NATURAL GAS INDEX: EP SAN JUAN - ICE FUTURES ENERGY DIV',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'NATURAL GAS HENRY LD1 FIXED - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PENULTIMATE ICE - ICE FUTURES ENERGY DIV',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
    'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-TMX WCS 1A INDEX - ICE FUTURES ENERGY DIV',
    'CRUDE DIFF-TMX SW 1A INDEX - ICE FUTURES ENERGY DIV',
    'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM - ICE FUTURES ENERGY DIV',
    'MT BELV NAT GASOLINE OPIS - NEW YORK MERCANTILE EXCHANGE',
     'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P FINANCIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P UTILITIES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P 400 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P ENERGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P CONSU STAPLES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P TECHNOLOGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P HEALTH CARE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE', 
     'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 ANNUAL DIVIDEND INDEX - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 QUARTERLY DIVIDEND IND - CHICAGO MERCANTILE EXCHANGE', 
     'DOW JONES U.S. REAL ESTATE IDX - CHICAGO BOARD OF TRADE',
     'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',  
     'NIKKEI STOCK AVERAGE YEN DENOM - CHICAGO MERCANTILE EXCHANGE',
     'COLUMBIA GULF TRANSMISSION CO. -  MAINLINE POOL - ICE FUTURES ENERGY DIV',
     'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COPPER-GRADE #1 - COMMODITY EXCHANGE INC.',
    'LITHIUM HYDROXIDE - COMMODITY EXCHANGE INC.',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
    'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MARINE FUEL OIL 0.5% FOB USGC - ICE FUTURES ENERGY DIV',
    'GULF JET NY HEAT OIL SPR - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
 'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM BARGES - ICE FUTURES ENERGY DIV',
    'EUR STYLE NATURAL GAS OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GAS LD1 for GDD -TEXOK - ICE FUTURES ENERGY DIV',
 'NAT GAS ICE LD1 - ICE FUTURES ENERGY DIV',
 'NATURAL GAS CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
 'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
 'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
 'BRENT LAST DAY - NEW YORK MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MICRO GOLD - COMMODITY EXCHANGE INC.',
    'WTI 1st Line-Brent 1st Line - ICE FUTURES ENERGY DIV',
    'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 TOTAL RETURN INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'NFX CS5TC CAPESIZE 5T/C AVG - NASDAQ FUTURES',
    'NFX PM4TC PANAMAX 4T/C AVG - NASDAQ FUTURES',
    'NORTHWEST PIPELINE - CANADIAN BORDER (BASIS) - ICE FUTURES ENERGY DIV',
    'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS CUSHING/WTI 1ST - ICE FUTURES ENERGY DIV',
    'DUTCH TTF NAT GAS CAL MONTH - NEW YORK MERCANTILE EXCHANGE',
    'TRANSCONTINENTAL GAS - ZONE 6 (NY) (BASIS) - ICE FUTURES ENERGY DIV',
    'GULF COAST UNL 87 GAS M2 PL RB - NEW YORK MERCANTILE EXCHANGE',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS INDEX: ALGONQUIN CITY GATES - ICE FUTURES ENERGY DIV',
    'HOT ROLLED COIL STEEL - NEW YORK MERCANTILE EXCHANGE',
    '3.5% FUEL OIL RDAM CRACK SPR - NEW YORK MERCANTILE EXCHANGE',
    'HENRY HUB PENULTIMATE NAT GAS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GASLNE OPIS MT B NONTET FP - ICE FUTURES ENERGY DIV',
     'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'SOUTH AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE',
    'Nano Bitcoin - LMX LABS LLC',
    'NANO ETHER - LMX LABS LLC',
    '2 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'WCS OIL NET ENERGY MONTHLY IND - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
    'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'WTI  HOUSTON ARGUS/WTI TR MO - NEW YORK MERCANTILE EXCHANGE',
    'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE',
    'HOUSTON SHIP CHANNEL (INDEX) - ICE FUTURES ENERGY DIV',
    'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE',
     'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
     'NAT GAS ICE PEN - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ERCOT Houston 345KV Hub RT 7x8 - ICE FUTURES ENERGY DIV',
    'ERCOT Houston 345KV RT OFF FIX - ICE FUTURES ENERGY DIV',
    'ERCOT HOUSTON 345KV RT PK FIX - ICE FUTURES ENERGY DIV',
     'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
 'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
  'GULF # 6 FUEL OIL CRACK - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) (BASIS) - ICE FUTURES ENERGY DIV',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) - ICE FUTURES ENERGY DIV',
 "WAHA HUB - WEST TEXAS DELIVERED/BUYER'S INDEX - ICE FUTURES ENERGY DIV",
 '5 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'MICRO 10 YEAR YIELD - CHICAGO BOARD OF TRADE', 
 'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
 'MICRO SING FOB MARINE FUEL .5% - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
 'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
 '5 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',
 'ARGUS WTI HOUSTON/WTI TRADE MO - ICE FUTURES ENERGY DIV',
 'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
 '10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '5-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '2-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 'UST BOND - CHICAGO BOARD OF TRADE',
 'ULTRA UST 10Y - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE',
 'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
 'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'FUEL OIL USGC HSFO PLATTS BALM - ICE FUTURES ENERGY DIV',
    'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'E-MINI S&P REAL ESTATE INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.',
    'EUROPEAN PROPANE CIF ARA - NEW YORK MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTYUNPAID - COMMODITY EXCHANGE INC.',
    'E-MINI S&P COMMUNICATION INDEX - CHICAGO MERCANTILE EXCHANGE',
    'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
    'RANDOM LENGTH LUMBER - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-3M - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-1M - CHICAGO MERCANTILE EXCHANGE',
    '1-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',
    'NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC',
    'NANO ETHER - COINBASE DERIVATIVES, LLC',
    'NANO ETHER PERP STYLE - COINBASE DERIVATIVES, LLC',
'NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.',
'Nano Bitcoin - COINBASE DERIVATIVES, LLC',
'RUSSELL 2000 ANNUAL DIVIDEND - CHICAGO MERCANTILE EXCHANGE',
'ULTRA US T BOND - CHICAGO BOARD OF TRADE',
'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
'WHEAT-HRSpring - MIAX FUTURES EXCHANGE',
'WTI  HOUSTON ARGUS/WTI BALMO - NEW YORK MERCANTILE EXCHANGE',
'3 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
'EURO FX - CHICAGO MERCANTILE EXCHANGE',
'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
'MICRO SOL - CHICAGO MERCANTILE EXCHANGE',
'MICRO XRP - CHICAGO MERCANTILE EXCHANGE',
'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC',
'NAT GAS TETCO-WLA INDEX - ICE FUTURES ENERGY DIV',
'GRP 3 SOC GAS VS RBOB SPR - NEW YORK MERCANTILE EXCHANGE',
'GOLD -1 TROY OUNCE - COINBASE DERIVATIVES'





    
    
]


In [58]:
for item in remove_items:
    if item in unique_markets:
        unique_markets.remove(item)

In [59]:
sorted(list(dict.fromkeys(unique_markets))) #this is the market list for making graphs.

['7 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'BITCOIN - CHICAGO MERCANTILE EXCHANGE',
 'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE',
 'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE',
 'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'COCOA - ICE FUTURES U.S.',
 'COFFEE C - ICE FUTURES U.S.',
 'COPPER- #1 - COMMODITY EXCHANGE INC.',
 'CORN - CHICAGO BOARD OF TRADE',
 'COTTON NO. 2 - ICE FUTURES U.S.',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
 'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE',
 'E-MINI S&P CONSUMER DISC INDEX - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE',
 'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE',
 'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE',
 'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE',
 'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE',
 'GOLD - COMMODITY EXCHANGE INC.',
 'GOLD -1 TROY OUNCE - COINBASE DERIVATIVES, LLC',
 'JAPANESE

In [60]:
#remove duplicates
unique_markets = list(dict.fromkeys(unique_markets))

### 4. Create Open Interest Index Value for a Commodity 

In [61]:
#create new DF for this part of the analysis
df2 = df.sort_values(['Market', 'Datetime'], ascending = [True, True])

In [62]:
#Group markerss and add Open Interest Index Column

group = df2.groupby("Market")["OI"]
df2["OI_Index"] = group.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

## Forumla for indexing:
#zi = (xi – min(x)) / (max(x) – min(x)) * 100 = (12 – 12) / (68 – 12) * 100 = 0


### 5. See if a market has a unique value based on your keyword or search term


In [63]:
def number_of_markets (keyword_or_phrase):
    
    filtered = df[df["Market"].str.contains(keyword_or_phrase, case=False)]
    filtered = filtered["Market"].unique().tolist()
    
    return filtered

In [64]:
number_of_markets("cocoa")

['COCOA - ICE FUTURES U.S.']

### Retail OI indexing

In [65]:
df2.columns.to_list()

['Market',
 'Datetime',
 'Date',
 'CFTC Contract Market Code',
 'CFTC Market Code in Initials',
 'CFTC Region Code',
 'CFTC Commodity Code',
 'OI',
 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
 'Open Interest (Old)',
 'Noncommercial Positions-Long (Old)',
 'Noncommercial Positions-Short (Old)',
 'Noncommercial Positions-Spreading (Old)',
 'Commercial Positions-Long (Old)',
 'Commercial Positions-Short (Old)',
 'Total Reportable Positions-Long (Old)',
 'Total Reportable Positions-Short (Old)',
 'Nonreportable Positions-Long (Old)',
 'Nonreportable Positions-Short (Old)',
 'Open Interest (Other)',
 'Noncommercial Positions-Long (Other)',
 'Noncommercial Positions-Short (Other)'

In [66]:
cols = ['Market', 'Datetime' , 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
        'OI_Index', "OI"]

In [67]:
df3 = df2[cols].copy()

In [68]:
row_test = df3.iloc[0, 1:10] #list of column headers and 1st row of data

In [69]:
df3["Net Retail Position"] = df3["Nonreportable Positions-Long (All)"] - df3["Nonreportable Positions-Short (All)"]

In [70]:
group2 = df3.groupby("Market")["Net Retail Position"]
df3["Retail_Index"] = group2.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [71]:
df3["Net Commercial Position"] = df3["Commercial Positions-Long (All)"] - df3["Commercial Positions-Short (All)"]


In [72]:
group3 = df3.groupby("Market")["Net Commercial Position"]
df3["Commercial_Index"] = group3.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [73]:
df3["Net Traders Position"] = df3["Noncommercial Positions-Long (All)"] - df3["Noncommercial Positions-Short (All)"]


In [74]:
group4 = df3.groupby("Market")["Net Traders Position"]
df3["Traders_Index"] = group4.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [75]:
df3.tail()

,Market,Datetime,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Noncommercial Positions-Spreading (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Total Reportable Positions-Long (All),Total Reportable Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All),OI_Index,OI,Net Retail Position,Retail_Index,Net Commercial Position,Commercial_Index,Net Traders Position,Traders_Index
68806,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-03-24,1591,1792,0,0,0,1591,1792,429,228,29.440688,2020,201,62.745098,0,NaN,-201,37.254902
68805,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-03-31,1176,1203,0,0,0,1176,1203,365,338,0.000000,1541,27,20.098039,0,NaN,-27,79.901961
68804,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-04-07,1177,1122,0,0,0,1177,1122,391,446,1.659496,1568,-55,0.000000,0,NaN,55,100.000000
68803,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-04-14,1912,2047,0,0,0,1912,2047,639,504,62.077443,2551,135,46.568627,0,NaN,-135,53.431373
68802,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-04-21,2549,2902,0,0,0,2549,2902,619,266,100.000000,3168,353,100.000000,0,NaN,-353,0.000000


### Summarise Latest Week's Data in a Table

In [76]:
OI_condition = df3[(df3['OI_Index'] >= 80) | (df3['OI_Index']<=20)]
Retail_condition = df3[(df3['Retail_Index'] >= 80) | (df3['Retail_Index']<=20)]
Commercial_condition = df3[(df3['Commercial_Index'] >= 80) | (df3['Commercial_Index']<=20)]
Date_coundition =df3["Datetime"].max() 

In [77]:
# Get the most recent date
most_recent_date = df3['Datetime'].max()

# Filter the DataFrame for the most recent date and conditions

summary_table = df3.loc[(df3['Datetime'] == most_recent_date) & 
                        (((df3['OI_Index'] >= 80) | (df3['OI_Index'] <= 20)) |
                         ((df3['Retail_Index'] >= 80) | (df3['Retail_Index'] <= 20)) |
                         ((df3['Commercial_Index'] >= 80) | (df3['Commercial_Index'] <= 20))),
                        ['Market', 'OI_Index', 'Retail_Index', 'Commercial_Index']]


In [78]:
summary_table_filtered = summary_table[summary_table['Market'].isin(unique_markets)]
summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_6195/3186465317.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


### IB Gateway Contract Mapping

In [79]:
# COT Market name → (IB_Symbol, IB_Exchange, IB_Currency)
ib_mapping = {
    # Metals
    'GOLD - COMMODITY EXCHANGE INC.':        ('GC',  'COMEX', 'USD'),
    'SILVER - COMMODITY EXCHANGE INC.':      ('SI',  'COMEX', 'USD'),
    'PLATINUM - NEW YORK MERCANTILE EXCHANGE': ('PL', 'NYMEX', 'USD'),
    'PALLADIUM - NEW YORK MERCANTILE EXCHANGE': ('PA', 'NYMEX', 'USD'),
    'COPPER- #1 - COMMODITY EXCHANGE INC.':  ('HG',  'COMEX', 'USD'),
    'MICRO GOLD - COMMODITY EXCHANGE INC.':  ('MGC', 'COMEX', 'USD'),
    'MICRO SILVER - COMMODITY EXCHANGE INC.': ('QI', 'COMEX', 'USD'),
    'MICRO COPPER - COMMODITY EXCHANGE INC.': ('MHG', 'COMEX', 'USD'),

    # Energies
    'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE':     ('CL', 'NYMEX', 'USD'),
    'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE':    ('RB', 'NYMEX', 'USD'),
    'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE':     ('NG', 'NYMEX', 'USD'),
    'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE': ('QG', 'NYMEX', 'USD'),

    # Grains
    'CORN - CHICAGO BOARD OF TRADE':       ('ZC', 'CBOT', 'USD'),
    'OATS - CHICAGO BOARD OF TRADE':       ('ZO', 'CBOT', 'USD'),
    'WHEAT-SRW - CHICAGO BOARD OF TRADE':  ('ZW', 'CBOT', 'USD'),
    'SOYBEANS - CHICAGO BOARD OF TRADE':   ('ZS', 'CBOT', 'USD'),
    'SOYBEAN MEAL - CHICAGO BOARD OF TRADE': ('ZM', 'CBOT', 'USD'),
    'SOYBEAN OIL - CHICAGO BOARD OF TRADE':  ('ZL', 'CBOT', 'USD'),
    'MINI SOYBEANS - CHICAGO BOARD OF TRADE': ('ZS', 'CBOT', 'USD'),

    # Softs
    'COCOA - ICE FUTURES U.S.':      ('CC', 'NYBOT', 'USD'),
    'COFFEE C - ICE FUTURES U.S.':   ('KC', 'NYBOT', 'USD'),
    'COTTON NO. 2 - ICE FUTURES U.S.': ('CT', 'NYBOT', 'USD'),
    'SUGAR NO. 11 - ICE FUTURES U.S.': ('SB', 'NYBOT', 'USD'),

    # Livestock
    'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE': ('GF', 'CME', 'USD'),
    'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE':     ('HE', 'CME', 'USD'),
    'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE':   ('LE', 'CME', 'USD'),

    # Indices
    'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE':  ('ES', 'CME', 'USD'),
    'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE':   ('RTY', 'CME', 'USD'),
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE': ('MNQ', 'CME', 'USD'),
    'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE': ('NKD', 'CME', 'USD'),
    'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE': ('RTY', 'CME', 'USD'),
    'VIX FUTURES - CBOE FUTURES EXCHANGE': ('VIX', 'CFE', 'USD'),

    # Currencies — ContFuture needs (ISO code, exchange, ccy, tradingClass=exchange symbol)
    'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE':  ('AUD', 'CME', 'USD', '6A'),
    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE':      ('GBP', 'CME', 'USD', '6B'),
    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE':    ('CAD', 'CME', 'USD', '6C'),
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE': ('EUR', 'CME', 'USD', '6E'),
    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE':       ('JPY', 'CME', 'USD', '6J'),
    'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE':       ('MXP', 'CME', 'USD', '6M'),
    'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': ('NZD', 'CME', 'USD', '6N'),
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE':        ('CHF', 'CME', 'USD', '6S'),
    'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE':    ('ZAR', 'CME', 'USD', '6Z'),
    'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE':     ('BRL', 'CME', 'USD', '6L'),

    # Crypto (CME futures, not spot)
    'BITCOIN - CHICAGO MERCANTILE EXCHANGE':        ('BRR', 'CME', 'USD'),
    'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE': ('ETH', 'CME', 'USD'),
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE':  ('MBT', 'CME', 'USD'),
    'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE':    ('MET', 'CME', 'USD'),

    # Financials
    'UST 2Y NOTE - CHICAGO BOARD OF TRADE':  ('ZT', 'CBOT', 'USD'),
    'UST 5Y NOTE - CHICAGO BOARD OF TRADE':  ('ZF', 'CBOT', 'USD'),
    'UST 10Y NOTE - CHICAGO BOARD OF TRADE': ('ZN', 'CBOT', 'USD'),
    'UST BOND - CHICAGO BOARD OF TRADE':     ('ZB', 'CBOT', 'USD'),
}

In [80]:
mapping_df = pd.DataFrame([
    {
        'Market': k,
        'IB_Symbol': v[3] if len(v) > 3 else v[0],
        'IB_Exchange': v[1],
        'IB_Currency': v[2],
        'IB_ContFutSymbol': v[0],
        'IB_TradingClass': v[3] if len(v) > 3 else '',
    }
    for k, v in ib_mapping.items()
])

In [81]:
market_categories = {
    'GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    'SILVER - COMMODITY EXCHANGE INC.': 'Metals',
    'PLATINUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'PALLADIUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'COPPER- #1 - COMMODITY EXCHANGE INC.': 'Metals',
    'SOYBEAN OIL - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEANS - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEAN MEAL - CHICAGO BOARD OF TRADE': 'Softs',
    'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'CORN - CHICAGO BOARD OF TRADE': 'Grains',
    'OATS - CHICAGO BOARD OF TRADE': 'Grains',
    'WHEAT-SRW - CHICAGO BOARD OF TRADE': 'Grains',
    'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'COCOA - ICE FUTURES U.S.': 'Softs',
    'COFFEE C - ICE FUTURES U.S.': 'Softs',
    'COTTON NO. 2 - ICE FUTURES U.S.': 'Softs',
    'SUGAR NO. 11 - ICE FUTURES U.S.': 'Softs',
    'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO SILVER - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO COPPER - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'UST 5Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 2Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 10Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST BOND - CHICAGO BOARD OF TRADE': 'Financials',
    'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
     'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE' : 'Currencies',
    'VIX FUTURES - CBOE FUTURES EXCHANGE': 'Indices',
     'MINI SOYBEANS - CHICAGO BOARD OF TRADE': 'Softs',
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE': 'Indices',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE': 'Energies'
}


In [82]:
#map the categories to the market in the mapping df
mapping_df["group"] = mapping_df["Market"].map(market_categories)

In [83]:
df4 = df3.copy()

In [84]:
df4 = pd.merge(df4, mapping_df, on="Market", how='left')

In [85]:
df4.rename(columns = {"Datetime" : "Date"}, inplace = True)

In [86]:
df4['group'].fillna('Financials', inplace=True)

/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_6195/638600763.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df4['group'].fillna('Financials', inplace=True)


## Create RSI Function


In [87]:
def rsi(data, periods=10):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

## Download Price Data from IB Gateway

In [88]:
def fetch_via_individual_contracts(ib, symbol, exchange, currency, start_date):
    """Fallback for symbols where ContFuture is unavailable (e.g. CME FX).
    Fetches individual expired+active contracts and stitches into a
    front-month continuous daily series.
    """
    from collections import defaultdict

    contract = Future(symbol=symbol, exchange=exchange, currency=currency)
    contract.includeExpired = True
    details = ib.reqContractDetails(contract)

    if not details:
        return []

    # Collect contracts whose expiry is relevant to our date range
    contracts = []
    for d in details:
        c = d.contract
        exp_str = c.lastTradeDateOrContractMonth
        try:
            exp_dt = date(int(exp_str[:4]), int(exp_str[4:6]),
                         int(exp_str[6:8]) if len(exp_str) >= 8 else 28)
        except Exception:
            continue
        if exp_dt >= start_date:
            contracts.append((exp_dt, c))

    contracts.sort(key=lambda x: x[0])

    # Request daily bars for each contract
    all_bars = []
    for exp_dt, c in contracts:
        end = datetime.combine(min(exp_dt + timedelta(days=1), date.today()),
                               datetime.min.time())
        bars = ib.reqHistoricalData(
            c, endDateTime=end, durationStr='1 Y',
            barSizeSetting='1 day', whatToShow='TRADES',
            useRTH=True, formatDate=1,
        )
        if bars:
            for b in bars:
                if b.date >= start_date:
                    all_bars.append((b.date, b, exp_dt))
        time.sleep(1)

    if not all_bars:
        return []

    # Stitch: for each date keep the bar from the nearest-expiry contract
    by_date = defaultdict(list)
    for bar_date, bar, exp in all_bars:
        by_date[bar_date].append((bar, exp))

    continuous = []
    for bar_date in sorted(by_date):
        candidates = by_date[bar_date]
        # Prefer the front-month (nearest expiry still >= bar_date)
        valid = [(b, e) for b, e in candidates if e >= bar_date]
        if valid:
            continuous.append(min(valid, key=lambda x: x[1])[0])
        else:
            continuous.append(max(candidates, key=lambda x: x[1])[0])

    return continuous

In [ ]:
from ib_insync import *
from datetime import date, datetime, timedelta
import time, os, json as _json

IB_HOST = '127.0.0.1'
IB_PORT = 4001       # live trading
CLIENT_ID = 1
START_DATE = date(2022, 1, 1)
DAILY_CACHE_FILE = 'ib_daily_cache.json'

# ── Load existing cache (if any) ──────────────────────────────────────
if os.path.exists(DAILY_CACHE_FILE):
    with open(DAILY_CACHE_FILE) as f:
        cached_records = _json.load(f)
    cache_df = pd.DataFrame(cached_records)
    cache_df['Date'] = pd.to_datetime(cache_df['Date'])
    last_dates = cache_df.groupby('Market')['Date'].max()
    print(f"Loaded cache: {len(cache_df)} rows across {cache_df['Market'].nunique()} markets")
else:
    cache_df = pd.DataFrame()
    last_dates = pd.Series(dtype='datetime64[ns]')
    print("No cache found — will do a full backfill from 2022")

# ── Connect to IB Gateway ─────────────────────────────────────────────
util.startLoop()
ib = IB()
ib.connect(IB_HOST, IB_PORT, clientId=CLIENT_ID)
print(f"Connected to IB Gateway: {ib.isConnected()}")

# ── Incremental fetch: only request days after last cached date ────────
failed_markets = []
new_rows = []

for market_name in unique_markets:
    row = mapping_df[mapping_df['Market'] == market_name]
    if row.empty:
        print(f"  ⚠ No IB mapping for {market_name}, skipping")
        failed_markets.append(market_name)
        continue

    display_sym    = row['IB_Symbol'].iloc[0]
    contfut_sym    = row['IB_ContFutSymbol'].iloc[0]
    exchange       = row['IB_Exchange'].iloc[0]
    currency       = row['IB_Currency'].iloc[0]
    trading_class  = row['IB_TradingClass'].iloc[0]

    if market_name in last_dates.index:
        fetch_from = last_dates[market_name].date() + timedelta(days=1)
    else:
        fetch_from = START_DATE

    if fetch_from >= date.today():
        print(f"  ✓ {market_name}: cache is up-to-date")
        continue

    days_needed = (date.today() - fetch_from).days
    print(f"  {market_name} ({display_sym}): fetching {days_needed} days from {fetch_from}...", end=" ")

    try:
        if trading_class:
            contract = ContFuture(symbol=contfut_sym, exchange=exchange,
                                  currency=currency, tradingClass=trading_class)
        else:
            contract = ContFuture(symbol=contfut_sym, exchange=exchange, currency=currency)
        qualified = ib.qualifyContracts(contract)

        if not qualified and not trading_class and exchange != 'GLOBEX':
            contract = ContFuture(symbol=contfut_sym, exchange='GLOBEX', currency=currency)
            qualified = ib.qualifyContracts(contract)

        if qualified:
            if days_needed > 365:
                duration = '5 Y'
            else:
                duration = f'{min(days_needed + 5, 365)} D'

            bars = ib.reqHistoricalData(
                contract, endDateTime='', durationStr=duration,
                barSizeSetting='1 day', whatToShow='TRADES',
                useRTH=True, formatDate=1,
            )
            kept = [b for b in bars if b.date >= fetch_from] if bars else []
        else:
            print(f"(individual contracts) ", end="")
            kept = fetch_via_individual_contracts(
                ib, display_sym, exchange, currency, fetch_from
            )
        if kept:
            for b in kept:
                new_rows.append({
                    'Date': pd.Timestamp(b.date),
                    'Open': b.open, 'High': b.high, 'Low': b.low,
                    'Close': b.close, 'Volume': b.volume,
                    'Market': market_name, 'IB_Symbol': display_sym,
                })
            print(f"✓ {len(kept)} new bars")
        else:
            print("no new bars")

    except Exception as e:
        print(f"✗ {e}")
        failed_markets.append(market_name)

    time.sleep(1)

print(f"\nFetched {len(new_rows)} new rows for {len(set(r['Market'] for r in new_rows))} markets")
if failed_markets:
    print(f"Failed / skipped: {failed_markets}")




Loaded cache: 44838 rows across 48 markets
Connected to IB Gateway: True
  GOLD - COMMODITY EXCHANGE INC. (GC): fetching 3 days from 2026-04-25... ✓ 1 new bars
  ⚠ No IB mapping for GOLD -1 TROY OUNCE - COINBASE DERIVATIVES, LLC, skipping
  SILVER - COMMODITY EXCHANGE INC. (SI): fetching 3 days from 2026-04-25... ✓ 1 new bars
  MICRO SILVER - COMMODITY EXCHANGE INC. (QI): fetching 3 days from 2026-04-25... ✓ 1 new bars
  PLATINUM - NEW YORK MERCANTILE EXCHANGE (PL): fetching 3 days from 2026-04-25... ✓ 1 new bars
  PALLADIUM - NEW YORK MERCANTILE EXCHANGE (PA): fetching 3 days from 2026-04-25... ✓ 1 new bars
  COPPER- #1 - COMMODITY EXCHANGE INC. (HG): fetching 3 days from 2026-04-25... ✓ 1 new bars
  MICRO COPPER - COMMODITY EXCHANGE INC. (MHG): fetching 3 days from 2026-04-25... ✓ 1 new bars
  SOYBEAN OIL - CHICAGO BOARD OF TRADE (ZL): fetching 3 days from 2026-04-25... ✓ 1 new bars
  NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE (NG): fetching 3 days from 2026-04-25... ✓ 1 new bars
  

Error 200, reqId 83: No security definition has been found for the request, contract: ContFuture(symbol='BRL', exchange='CME', currency='USD', tradingClass='6L')
Unknown contract: ContFuture(symbol='BRL', exchange='CME', currency='USD', tradingClass='6L')
Error 200, reqId 84: No security definition has been found for the request, contract: Future(symbol='6L', exchange='CME', currency='USD', includeExpired=True)


  BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE (6L): fetching 1578 days from 2022-01-01... (individual contracts) no new bars
  SWISS FRANC - CHICAGO MERCANTILE EXCHANGE (6S): fetching 3 days from 2026-04-25... ✓ 1 new bars
  CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE (6C): fetching 3 days from 2026-04-25... ✓ 1 new bars
  BRITISH POUND - CHICAGO MERCANTILE EXCHANGE (6B): fetching 3 days from 2026-04-25... ✓ 1 new bars
  JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE (6J): fetching 3 days from 2026-04-25... ✓ 1 new bars
  NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE (6N): fetching 3 days from 2026-04-25... ✓ 1 new bars
  SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE (6Z): fetching 3 days from 2026-04-25... ✓ 1 new bars
  BITCOIN - CHICAGO MERCANTILE EXCHANGE (BRR): fetching 3 days from 2026-04-25... ✓ 1 new bars


Error 200, reqId 99: No security definition has been found for the request, contract: ContFuture(symbol='ETH', exchange='CME', currency='USD')
Unknown contract: ContFuture(symbol='ETH', exchange='CME', currency='USD')
Error 200, reqId 100: No security definition has been found for the request, contract: ContFuture(symbol='ETH', exchange='GLOBEX', currency='USD')
Unknown contract: ContFuture(symbol='ETH', exchange='GLOBEX', currency='USD')
Error 200, reqId 101: No security definition has been found for the request, contract: Future(symbol='ETH', exchange='CME', currency='USD', includeExpired=True)


  ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE (ETH): fetching 1578 days from 2022-01-01... (individual contracts) no new bars
  ⚠ No IB mapping for 7 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE, skipping
  VIX FUTURES - CBOE FUTURES EXCHANGE (VIX): fetching 3 days from 2026-04-25... ✓ 1 new bars

Fetched 52 new rows for 48 markets
Failed / skipped: ['GOLD -1 TROY OUNCE - COINBASE DERIVATIVES, LLC', 'E-MINI S&P CONSUMER DISC INDEX - CHICAGO MERCANTILE EXCHANGE', '7 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE']


Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: eufarm; secdefeu. The following farms are not connected: euhmds; ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: eufarm; secdefeu. The following farms are not connected: euhmds; ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: eufarm; secdefeu. The following farms are not connected: euhmds; ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 11

In [90]:
# Merge new rows into cache and persist to disk
if new_rows:
    new_df = pd.DataFrame(new_rows)
    daily_price_df = pd.concat([cache_df, new_df], ignore_index=True)
    daily_price_df = daily_price_df.drop_duplicates(subset=['Market', 'Date'], keep='last')
    daily_price_df = daily_price_df.sort_values(['Market', 'Date']).reset_index(drop=True)
else:
    daily_price_df = cache_df.copy()

# Save updated cache
save_df = daily_price_df.copy()
save_df['Date'] = save_df['Date'].dt.strftime('%Y-%m-%d')
with open(DAILY_CACHE_FILE, 'w') as f:
    _json.dump(save_df.to_dict('records'), f)

# Restore Date as Timestamp for the rest of the notebook
daily_price_df['Date'] = pd.to_datetime(daily_price_df['Date'])

print(f"✓ Daily price cache: {len(daily_price_df)} records across {daily_price_df['Market'].nunique()} markets")
print(f"✓ Date range: {daily_price_df['Date'].min().date()} → {daily_price_df['Date'].max().date()}")
print(f"✓ Saved to {DAILY_CACHE_FILE}")

✓ Daily price cache: 44890 records across 48 markets
✓ Date range: 2022-01-03 → 2026-04-28
✓ Saved to ib_daily_cache.json


In [91]:
# DIAGNOSTIC: Identify markets with missing price data (only unique_markets)
print("=" * 80)
print("PRICE DATA DIAGNOSTIC - Missing Data by Market")
print("=" * 80)

# Check which markets in unique_markets have no price data at all
target_markets = set(unique_markets)
price_markets = set(daily_price_df['Market'].unique())
missing_entirely = target_markets - price_markets

print(f"\n🔴 Markets in unique_markets but NO price data ({len(missing_entirely)}):")
for m in sorted(missing_entirely):
    print(f"   - {m}")

# Check for markets with partial/null price data (only unique_markets)
print(f"\n🟡 Markets with incomplete OHLC data:")
# Filter to only unique_markets
df_filtered = daily_price_df[daily_price_df['Market'].isin(unique_markets)]
null_summary = df_filtered.groupby('Market').agg({
    'Close': lambda x: x.isnull().sum(),
    'Open': lambda x: x.isnull().sum(),
    'High': lambda x: x.isnull().sum(),
    'Low': lambda x: x.isnull().sum(),
    'Volume': lambda x: x.isnull().sum()
}).rename(columns={'Close': 'Close_nulls', 'Open': 'Open_nulls', 
                   'High': 'High_nulls', 'Low': 'Low_nulls', 'Volume': 'Volume_nulls'})

# Also get total rows per market
null_summary['Total_Rows'] = df_filtered.groupby('Market').size()

# Show markets with any null values
markets_with_nulls = null_summary[
    (null_summary['Close_nulls'] > 0) | 
    (null_summary['Open_nulls'] > 0) | 
    (null_summary['High_nulls'] > 0) | 
    (null_summary['Low_nulls'] > 0)
]

if len(markets_with_nulls) > 0:
    print(markets_with_nulls.to_string())
else:
    print("   ✓ All markets have complete OHLC data!")

# Summary stats
print(f"\n📊 Summary:")
print(f"   Total unique_markets: {len(target_markets)}")
print(f"   Markets with price data: {len(price_markets)}")
print(f"   Markets missing entirely: {len(missing_entirely)}")
print("=" * 80)


PRICE DATA DIAGNOSTIC - Missing Data by Market

🔴 Markets in unique_markets but NO price data (5):
   - 7 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE
   - BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE
   - E-MINI S&P CONSUMER DISC INDEX - CHICAGO MERCANTILE EXCHANGE
   - ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE
   - GOLD -1 TROY OUNCE - COINBASE DERIVATIVES, LLC

🟡 Markets with incomplete OHLC data:
   ✓ All markets have complete OHLC data!

📊 Summary:
   Total unique_markets: 53
   Markets with price data: 48
   Markets missing entirely: 5


#### Add Price data to COT dataframe

In [92]:
df5 = pd.merge(df4, daily_price_df[['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume']], on=['Market', 'Date'], how='left')

##### df 5 now contains price and open interest by week but for all markets, not just the unique_markets

In [93]:
df5 = df5.sort_values(['Market', 'Date'], ascending = [True, True])

##### df6 has weekly COT and price data along with RSI but does not have daily prices

In [94]:
#Put the data above into a dataframe without the COT data for markets we are not trading (i.e removing everything not in unique markets)

df6 = []  # Initialize an empty list to store filtered dataframes

for market_name in unique_markets:
    df6.append(df5[df5["Market"] == market_name])

# Concatenate the filtered dataframes into a single DataFrame
df6 = pd.concat(df6, ignore_index=True)


        

In [95]:
# Calculate RSI for daily price data
daily_price_df['RSI'] = daily_price_df.groupby('Market')['Close'].transform(lambda x: rsi(x))

# Add data_type to existing df6 (weekly COT data)
df6_weekly = df6.copy()
df6_weekly['data_type'] = 'weekly_cot'

# Prepare daily price data to match df6 structure
daily_price_expanded = daily_price_df.copy()
daily_price_expanded['data_type'] = 'daily_price'

cot_columns = ['OI', 'OI_Index', 'Retail_Index', 'Commercial_Index', 'Traders_Index', 
               'Net Retail Position', 'Net Commercial Position', 'Net Traders Position',
               'Noncommercial Positions-Long (All)', 'Noncommercial Positions-Short (All)',
               'Commercial Positions-Long (All)', 'Commercial Positions-Short (All)',
               'Nonreportable Positions-Long (All)', 'Nonreportable Positions-Short (All)']

for col in cot_columns:
    if col not in daily_price_expanded.columns:
        daily_price_expanded[col] = None

if 'group' not in daily_price_expanded.columns:
    daily_price_expanded = pd.merge(daily_price_expanded, mapping_df[['Market', 'group']], on='Market', how='left')

if 'IB_Symbol' not in df6_weekly.columns:
    df6_weekly = pd.merge(df6_weekly, mapping_df[['Market', 'IB_Symbol']], on='Market', how='left')

common_columns = ['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'RSI', 'IB_Symbol', 'data_type', 'group'] + cot_columns

df6_weekly = df6_weekly.reindex(columns=common_columns, fill_value=None)
daily_price_expanded = daily_price_expanded.reindex(columns=common_columns, fill_value=None)

print(f"✓ Common columns: {common_columns[:12]}...")

df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)
df6_combined = df6_combined.sort_values(['Market', 'Date']).reset_index(drop=True)

print(f"Combined dataset: {len(df6_combined)} total records")
print(f"  Weekly COT: {len(df6_weekly)} | Daily price: {len(daily_price_expanded)}")


✓ Common columns: ['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'RSI', 'IB_Symbol', 'data_type', 'group', 'OI']...


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_6195/1487681820.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)


Combined dataset: 54519 total records
  Weekly COT: 9629 | Daily price: 44890


In [96]:
df6_combined = df6_combined.dropna(subset=['Close'])

In [97]:
# Filter for March 9th, 2026 only
df_march_9 = df6_combined[df6_combined['Date'] == '2026-03-10']

markets_with_close = df_march_9[df_march_9['Close'].notna()]['Market'].unique()

print(f"Total markets with close prices on March 10th: {len(markets_with_close)}")
print("\nMarkets with close prices on March 10th:")
for market in sorted(markets_with_close):
    print(f"  - {market}")

Total markets with close prices on March 10th: 48

Markets with close prices on March 10th:
  - AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  - BITCOIN - CHICAGO MERCANTILE EXCHANGE
  - BRITISH POUND - CHICAGO MERCANTILE EXCHANGE
  - CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  - COCOA - ICE FUTURES U.S.
  - COFFEE C - ICE FUTURES U.S.
  - COPPER- #1 - COMMODITY EXCHANGE INC.
  - CORN - CHICAGO BOARD OF TRADE
  - COTTON NO. 2 - ICE FUTURES U.S.
  - E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE
  - E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE
  - EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE
  - EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE
  - FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE
  - GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE
  - GOLD - COMMODITY EXCHANGE INC.
  - JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE
  - LEAN HOGS - CHICAGO MERCANTILE EXCHANGE
  - LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE
  - MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE
  -

In [98]:
platinum = df6_combined[df6_combined["Market"] == "SILVER - COMMODITY EXCHANGE INC."]
platinum.sort_values("Date", ascending = False)

,Market,Date,Close,Open,High,Low,Volume,RSI,IB_Symbol,data_type,...,Traders_Index,Net Retail Position,Net Commercial Position,Net Traders Position,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All)
39767,SILVER - COMMODITY EXCHANGE INC.,2026-04-27,75.025,75.465,75.705,74.590,8281.0,48.367347,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
39766,SILVER - COMMODITY EXCHANGE INC.,2026-04-24,76.415,75.425,76.665,75.175,10198.0,49.829172,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
39765,SILVER - COMMODITY EXCHANGE INC.,2026-04-23,75.505,76.025,76.520,74.585,14770.0,47.424952,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
39764,SILVER - COMMODITY EXCHANGE INC.,2026-04-22,77.960,78.030,78.470,77.575,8423.0,57.684273,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
39763,SILVER - COMMODITY EXCHANGE INC.,2026-04-21,76.485,78.475,79.110,75.380,23107.0,62.976501,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38561,SILVER - COMMODITY EXCHANGE INC.,2022-05-04,26.860,26.860,26.860,26.860,0.0,NaN,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
38559,SILVER - COMMODITY EXCHANGE INC.,2022-05-03,27.100,27.100,27.105,27.100,0.0,NaN,SI,weekly_cot,...,51.091823,11249,-39317,28068,56764,28696,50184,89501,23860,12611
38560,SILVER - COMMODITY EXCHANGE INC.,2022-05-03,27.100,27.100,27.105,27.100,0.0,NaN,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
38558,SILVER - COMMODITY EXCHANGE INC.,2022-05-02,26.970,26.970,26.970,26.965,0.0,NaN,SI,daily_price,...,NaN,None,None,None,None,None,None,None,None,None


In [99]:
# Export the combined dataset (both weekly COT data and daily price data)
df6_combined.to_json('cot_data.json', orient='records')
print("Combined data (weekly COT + daily prices) exported to cot_data.json")
print(f"Total records exported: {len(df6_combined)}")

Combined data (weekly COT + daily prices) exported to cot_data.json
Total records exported: 52785


In [100]:
# =============================================================================
# 🔔 CHECK FOR RECENT TRADING SIGNALS (Last 7 Days)
# =============================================================================
from datetime import datetime, timedelta

def check_recent_signals(df, days=7, commercial_long=80, commercial_short=20, 
                         rsi_oversold=30, rsi_overbought=70):
    """Check for trading signals in the last N days."""
    cutoff_date = datetime.now() - timedelta(days=days)
    recent_signals = []
    
    markets = df['Market'].unique()
    
    for market in markets:
        market_data = df[df['Market'] == market].copy()
        price_daily = market_data[market_data['data_type'] == 'daily_price'].copy()
        cot_weekly = market_data[market_data['data_type'] == 'weekly_cot'].copy()
        
        if price_daily.empty or cot_weekly.empty:
            continue
        
        # Check if Commercial_Index exists in COT data
        if 'Commercial_Index' not in cot_weekly.columns:
            continue
            
        # Sort both by Date for merge_asof
        price_daily = price_daily[['Date', 'Close', 'RSI']].copy().sort_values('Date').reset_index(drop=True)
        cot_for_merge = cot_weekly[['Date', 'Commercial_Index']].dropna(subset=['Commercial_Index']).copy()
        cot_for_merge = cot_for_merge.sort_values('Date').reset_index(drop=True)
        
        if cot_for_merge.empty:
            continue
        
        # Use merge_asof to carry forward most recent COT data to each daily price row
        # direction='backward' means: find most recent COT date <= each price date
        merged = pd.merge_asof(price_daily, cot_for_merge, on='Date', direction='backward')
        
        # Drop rows with missing required data
        merged = merged.dropna(subset=['Close', 'RSI', 'Commercial_Index'])
        
        # Filter to recent days
        merged = merged[merged['Date'] >= cutoff_date]
        
        for _, row in merged.iterrows():
            signal_type = None
            # Long: Commercial >= 80 AND RSI < 30
            if row['Commercial_Index'] >= commercial_long and row['RSI'] < rsi_oversold:
                signal_type = 'LONG'
            # Short: Commercial <= 20 AND RSI > 70
            elif row['Commercial_Index'] <= commercial_short and row['RSI'] > rsi_overbought:
                signal_type = 'SHORT'
            
            if signal_type:
                recent_signals.append({
                    'Market': market,
                    'Date': row['Date'],
                    'Signal': signal_type,
                    'COT': round(row['Commercial_Index'], 1),
                    'RSI': round(row['RSI'], 1),
                    'Close': round(row['Close'], 2)
                })
    
    return sorted(recent_signals, key=lambda x: x['Date'], reverse=True)

# Check for signals
signals = check_recent_signals(df6_combined, days=7)

print("=" * 70)
if signals:
    print(f"🔔 ALERT: {len(signals)} TRADING SIGNAL(S) IN THE LAST 7 DAYS!")
    print("=" * 70)
    for sig in signals:
        emoji = "🟢" if sig['Signal'] == 'LONG' else "🔴"
        print(f"{emoji} {sig['Signal']:5} | {sig['Date'].strftime('%Y-%m-%d')} | {sig['Market'][:40]}")
        print(f"         COT: {sig['COT']}, RSI: {sig['RSI']}, Price: ${sig['Close']:,.2f}")
        print("-" * 70)
else:
    print("✓ No new trading signals in the last 7 days")
    print("=" * 70)


🔔 ALERT: 19 TRADING SIGNAL(S) IN THE LAST 7 DAYS!
🔴 SHORT | 2026-04-27 | AUSTRALIAN DOLLAR - CHICAGO MERCANTILE E
         COT: 7.0, RSI: 71.0, Price: $0.72
----------------------------------------------------------------------
🔴 SHORT | 2026-04-27 | EMINI RUSSELL 1000 GROWTH - CHICAGO MERC
         COT: 0.0, RSI: 79.9, Price: $2,799.60
----------------------------------------------------------------------
🔴 SHORT | 2026-04-27 | SOYBEAN OIL - CHICAGO BOARD OF TRADE
         COT: 0.0, RSI: 79.1, Price: $71.67
----------------------------------------------------------------------
🔴 SHORT | 2026-04-27 | SOYBEANS - CHICAGO BOARD OF TRADE
         COT: 10.7, RSI: 70.7, Price: $1,192.00
----------------------------------------------------------------------
🔴 SHORT | 2026-04-24 | AUSTRALIAN DOLLAR - CHICAGO MERCANTILE E
         COT: 7.0, RSI: 70.9, Price: $0.71
----------------------------------------------------------------------
🔴 SHORT | 2026-04-24 | BITCOIN - CHICAGO MERCANTILE EXCHANGE


In [101]:
df6_combined.iloc[-1]

Market                                 WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE
Date                                                           2026-04-27 00:00:00
Close                                                                        96.37
Open                                                                         95.83
High                                                                         97.67
Low                                                                           95.3
Volume                                                                     86542.0
RSI                                                                      45.917445
IB_Symbol                                                                       CL
data_type                                                              daily_price
group                                                                     Energies
OI                                                                            None
OI_I

## Build Intraday Cache from IB (30m / 60m)

Populates `ORB_intraday_data.json` so that `Inside_day_backtest.py` and `ORB_backtest.py` can read from cache instead of calling Yahoo Finance.

IB limits: 30m data goes back ~30-60 days only. Run this cell periodically to keep the cache fresh.

In [102]:
# Incrementally update ORB_intraday_data.json with 30m / 60m bars from IB
INTRADAY_CACHE_FILE = 'ORB_intraday_data.json'

# Load existing cache
if os.path.exists(INTRADAY_CACHE_FILE):
    with open(INTRADAY_CACHE_FILE) as f:    
        existing = _json.load(f)
    intra_cache = pd.DataFrame(existing)
    if not intra_cache.empty:
        intra_cache['datetime'] = pd.to_datetime(intra_cache['datetime'])
    print(f"Loaded intraday cache: {len(intra_cache)} rows")
else:
    intra_cache = pd.DataFrame()
    print("No intraday cache — building from scratch")

new_intra = []
for market_name in unique_markets:
    row = mapping_df[mapping_df['Market'] == market_name]
    if row.empty:
        continue

    contfut_sym    = row['IB_ContFutSymbol'].iloc[0]
    exchange       = row['IB_Exchange'].iloc[0]
    currency       = row['IB_Currency'].iloc[0]
    trading_class  = row['IB_TradingClass'].iloc[0]

    try:
        if trading_class:
            contract = ContFuture(symbol=contfut_sym, exchange=exchange,
                                  currency=currency, tradingClass=trading_class)
        else:
            contract = ContFuture(symbol=contfut_sym, exchange=exchange, currency=currency)
        qualified = ib.qualifyContracts(contract)
        if not qualified:
            print(f"  ✗ {market_name}: could not qualify")
            continue

        for interval in ['30m', '60m']:
            bar_size = '30 mins' if interval == '30m' else '1 hour'

            # Check cache for this symbol+interval to determine how far back to fetch
            last_cached = None
            if not intra_cache.empty:
                mask = (intra_cache['symbol'] == market_name) & (intra_cache['interval'] == interval)
                cached_subset = intra_cache.loc[mask]
                if not cached_subset.empty:
                    last_cached = cached_subset['datetime'].max()

            if last_cached is not None:
                gap_days = (pd.Timestamp.now() - last_cached).days + 1
                if gap_days <= 1:
                    print(f"  ⏭ {market_name} {interval}: cache is current ({last_cached.date()})")
                    continue
                duration = f'{min(gap_days, 365)} D'
            else:
                duration = '1 Y'

            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr=duration,
                barSizeSetting=bar_size,
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1,
            )
            if bars:
                for b in bars:
                    new_intra.append({
                        'symbol': market_name,
                        'interval': interval,
                        'datetime': b.date,
                        'open': b.open, 'high': b.high,
                        'low': b.low, 'close': b.close,
                        'volume': b.volume,
                    })
                print(f"  ✓ {market_name} {interval}: {len(bars)} bars")
            else:
                print(f"  ⚠ {market_name} {interval}: no data")
            time.sleep(2)

    except Exception as e:
        print(f"  ✗ {market_name}: {e}")
    time.sleep(1)

# Merge new with existing, dedup on (symbol, interval, datetime)
if new_intra:
    new_df = pd.DataFrame(new_intra)
    new_df['datetime'] = pd.to_datetime(new_df['datetime'], utc=True).dt.tz_localize(None)
    combined = pd.concat([intra_cache, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=['symbol', 'interval', 'datetime'], keep='last')
    combined = combined.sort_values(['symbol', 'interval', 'datetime']).reset_index(drop=True)
else:
    combined = intra_cache.copy()

# Save
save_intra = combined.copy()
save_intra['datetime'] = save_intra['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
with open(INTRADAY_CACHE_FILE, 'w') as f:
    _json.dump(save_intra.to_dict('records'), f)

print(f"\n✓ Intraday cache: {len(combined)} total rows → {INTRADAY_CACHE_FILE}")

Loaded intraday cache: 771752 rows
  ✗ GOLD - COMMODITY EXCHANGE INC.: Not connected
  ✗ SILVER - COMMODITY EXCHANGE INC.: Not connected
  ✗ MICRO SILVER - COMMODITY EXCHANGE INC.: Not connected
  ✗ PLATINUM - NEW YORK MERCANTILE EXCHANGE: Not connected
  ✗ PALLADIUM - NEW YORK MERCANTILE EXCHANGE: Not connected
  ✗ COPPER- #1 - COMMODITY EXCHANGE INC.: Not connected
  ✗ MICRO COPPER - COMMODITY EXCHANGE INC.: Not connected
  ✗ SOYBEAN OIL - CHICAGO BOARD OF TRADE: Not connected
  ✗ NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE: Not connected
  ✗ GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE: Not connected
  ✗ COCOA - ICE FUTURES U.S.: Not connected
  ✗ CORN - CHICAGO BOARD OF TRADE: Not connected
  ✗ OATS - CHICAGO BOARD OF TRADE: Not connected
  ✗ WHEAT-SRW - CHICAGO BOARD OF TRADE: Not connected
  ✗ SOYBEAN MEAL - CHICAGO BOARD OF TRADE: Not connected
  ✗ SOYBEANS - CHICAGO BOARD OF TRADE: Not connected
  ✗ MINI SOYBEANS - CHICAGO BOARD OF TRADE: Not connected
  ✗ FEEDER CATTLE - CHICA

In [52]:
ib.disconnect()
print("Disconnected from IB Gateway")

Disconnected from IB Gateway
